<a href="https://colab.research.google.com/github/samuelezranas/morse_model/blob/main/morse_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Instalasi Library

In [ ]:
!pip install pydub librosa tensorflow

Generator Dataset Sintetis

In [ ]:
import numpy as np
from pydub import AudioSegment
from pydub.generators import Sine
import os
import random

# Mapping Morse (Logika dari Kaggle)
MORSE_CODE_DICT = {'A': '.-', 'B': '-...', 'C': '-.-.', 'D': '-..', 'E': '.', 'F': '..-.', 'G': '--.', 'H': '....', 'I': '..', 'J': '.---', 'K': '-.-', 'L': '.-..', 'M': '--', 'N': '-.', 'O': '---', 'P': '.--.', 'Q': '--.-', 'R': '.-.', 'S': '...', 'T': '-', 'U': '..-', 'V': '...-', 'W': '.--', 'X': '-..-', 'Y': '-.--', 'Z': '--..'}

def generate_morse_audio(morse_str, filename, wpm=20, fs=16000):
    dot_duration = 1200 / wpm  # Durasi titik dalam ms
    audio = AudioSegment.silent(duration=100)

    for symbol in morse_str:
        if symbol == '.':
            audio += Sine(600).to_audio_segment(duration=dot_duration)
        elif symbol == '-':
            audio += Sine(600).to_audio_segment(duration=dot_duration * 3)
        audio += AudioSegment.silent(duration=dot_duration) # Spasi antar simbol

    audio.export(filename, format="wav")

# Membuat folder dataset
os.makedirs('dataset/dot', exist_ok=True)
os.makedirs('dataset/dash', exist_ok=True)

# Generate 500 sampel titik dan 500 sampel garis dengan sedikit variasi durasi/pitch
for i in range(500):
    generate_morse_audio('.', f'dataset/dot/dot_{i}.wav', wpm=random.randint(15, 25))
    generate_morse_audio('-', f'dataset/dash/dash_{i}.wav', wpm=random.randint(15, 25))

print("Dataset berhasil dibuat langsung di Colab!")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Dataset berhasil dibuat langsung di Colab!


In [ ]:
!pip install resampy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 33.2 MB/s eta 0:00:00


In [ ]:
import librosa

def extract_features(file_path):
    audio, sample_rate = librosa.load(file_path, res_type='soxr_hq')
    mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
    return np.mean(mfccs.T, axis=0)

features = []
labels = []

for label_kind in ['dot', 'dash']:
    folder = f'dataset/{label_kind}'
    for filename in os.listdir(folder):
        data = extract_features(os.path.join(folder, filename))
        features.append(data)
        labels.append(0 if label_kind == 'dot' else 1)

X = np.array(features)
y = np.array(labels)
print("Fitur berhasil diekstrak.")

Fitur berhasil diekstrak.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Dense(128, activation='relu', input_shape=(40,)),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dense(2, activation='softmax') # Output: Dot atau Dash
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X, y, epochs=20, batch_size=32, validation_split=0.2)

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.6187 - loss: 4.4210 - val_accuracy: 1.0000 - val_loss: 0.0069
Epoch 2/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8575 - loss: 0.9023 - val_accuracy: 1.0000 - val_loss: 0.0011
Epoch 3/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9375 - loss: 0.3083 - val_accuracy: 1.0000 - val_loss: 6.5397e-06
Epoch 4/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9625 - loss: 0.1371 - val_accuracy: 1.0000 - val_loss: 6.7949e-08
Epoch 5/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9638 - loss: 0.1260 - val_accuracy: 1.0000 - val_loss: 3.3379e-08
Epoch 6/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9812 - loss: 0.0890 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 7/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9862 - loss: 0.0282 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 8/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9962 - loss: 0.0208 - val_accuracy: 1.0000 -

In [ ]:
# Convert model ke TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Simpan file
with open('morse_model.tflite', 'wb') as f:
    f.write(tflite_model)

print("Model TFLite siap diunduh untuk aplikasi Android & WearOS kamu!")

Saved artifact at '/tmp/tmpqagu077j'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 40), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  132740502015056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132740502016592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132740502018704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132740502017360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132740502017744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132740502019280: TensorSpec(shape=(), dtype=tf.resource, name=None)
Model TFLite siap diunduh untuk aplikasi Android & WearOS kamu!
